# 01 – CD8αα(I) → DC1/aDC1, fetal versus postnatal: the cells

**Why this case.** In thymus case 2 (`notebooks/thymus_case2_xcl1`), NK and CD8αα(I) cells were pooled as XCL1
senders. XCL1 in CD8αα(I) cells fell with age (36% of cells at 7–11 PCW, 10% from 15 months), while it rose in NK
cells, and the pooled network showed no XCL1–age link (case 2, notebook 05). Here the senders are **CD8αα(I) cells
only**, the cell type named in the claim, and the fetal and postnatal thymus are analysed in **two separate runs**.

**Question of this notebook.** How many cells does each run have, where do they come from, and how does XCL1 in
CD8αα(I) cells change with age, donor by donor?

Decisions:
- **senders** = CD8αα(I) (group `cd8aa`); **receivers** = DC1 + aDC1 (group `dc1`), as in case 2;
- **age class**: fetal = ages in post-conception weeks (7–17 PCW), postnatal = 3 months to 35 years;
- counts from the raw-count h5ad, labels from the fig1 `.obs`; expression levels on the authors' scale (fig1 matrix).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path("../..")
THY = ROOT / "data/thymus"
OUT = ROOT / "results/thymus_case3_xcl1_age/01_cell_populations"
OUT.mkdir(parents=True, exist_ok=True)

annot = ad.read_h5ad(THY / "thymus_annotated_matrix_files/HTA08.v01.A05.Science_human_fig1.h5ad", backed="r")
raw = ad.read_h5ad(THY / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")

SENDERS = ["CD8αα(I)"]
RECEIVERS = ["DC1", "aDC1"]
ORDER = SENDERS + RECEIVERS
GROUP = {"CD8αα(I)": "cd8aa", "DC1": "dc1", "aDC1": "dc1"}

def age_in_weeks(a):
    v = float(a[:-1])
    return v if a.endswith("w") else 40 + v * 4.35 if a.endswith("m") else 40 + v * 52

cells = annot.obs[["Anno_level_5", "Anno_stage", "donor", "Age", "method", "sort", "Source"]].astype(str)
cells = cells.join(raw.obs[["n_counts", "n_genes", "mito"]])
cells["age_class"] = np.where(cells.Age.str.endswith("w"), "fetal", "postnatal")

sel = cells[cells.Anno_level_5.isin(ORDER)].copy()
sel["group"] = sel.Anno_level_5.map(GROUP)
pd.crosstab([sel.group, sel.Anno_level_5], sel.age_class, margins=True)

/tmp/ipykernel_411/3280087176.py:14: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  annot = ad.read_h5ad(THY / "thymus_annotated_matrix_files/HTA08.v01.A05.Science_human_fig1.h5ad", backed="r")


/tmp/ipykernel_411/3280087176.py:15: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  raw = ad.read_h5ad(THY / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")


age_class           fetal  postnatal   All
group Anno_level_5                        
cd8aa CD8αα(I)       1469       2430  3899
dc1   DC1            1071        312  1383
      aDC1            100         35   135
All                  2640       2777  5417

In [2]:
# Donors and chemistry per age class
pd.concat({"donors": sel.groupby(["Anno_level_5", "age_class"]).donor.nunique().unstack(),
           "3GEX": sel[sel.method == "3GEX"].groupby(["Anno_level_5", "age_class"]).size().unstack(),
           "5GEX": sel[sel.method == "5GEX"].groupby(["Anno_level_5", "age_class"]).size().unstack()},
          axis=1).reindex(ORDER).fillna(0).astype(int)

donors            3GEX            5GEX          
age_class     fetal postnatal fetal postnatal fetal postnatal
Anno_level_5                                                 
CD8αα(I)         14         8   484       952   985      1478
DC1              15         8   733       102   338       210
aDC1             11         6    37         3    63        32

**Reading.**
- CD8αα(I) has enough cells in both age classes: 1,469 fetal cells from 14 donors and 2,430 postnatal cells from
  8 donors.
- Receivers:
  - DC1 is mainly fetal (1,071 against 312 postnatal);
  - aDC1 is small in both: 100 fetal cells, and **only 35 postnatal cells, from 6 donors**, almost all 5′ (32 of 35).
  - The postnatal aDC1 part of the receiver network will therefore be weak.
- Chemistry differs between the age classes for the receivers (DC1 is 32% 5′ in fetal, 67% in postnatal), so `method`
  stays a network variable in both runs.

## 1. XCL1 and XCL2 in CD8αα(I) cells, donor by donor

Fraction of CD8αα(I) cells with XCL1 (and XCL2), and mean expression on the authors' scale (log1p of counts per
100,000, fig1 matrix), per donor ordered by age. Only donors with at least 20 CD8αα(I) cells are shown.

In [3]:
GENES = ["XCL1", "XCL2", "GNG4", "XCR1", "CCR7"]
gene_idx = []
for g in GENES:
    hits = np.flatnonzero(annot.var_names == g)
    assert len(hits) == 1, g
    gene_idx.append(hits[0])

rows = np.sort(annot.obs_names.get_indexer(sel.index))
logexpr = pd.DataFrame(np.asarray(annot[rows].X[:, gene_idx].toarray()), index=annot.obs_names[rows], columns=GENES)
sel = sel.loc[logexpr.index]

cd8 = sel[sel.Anno_level_5 == "CD8αα(I)"].join(logexpr)
per_donor = cd8.groupby("donor").agg(
    Age=("Age", "first"), age_class=("age_class", "first"), cells=("XCL1", "size"),
    XCL1_detected=("XCL1", lambda x: (x > 0).mean()), XCL1_mean=("XCL1", "mean"),
    XCL2_detected=("XCL2", lambda x: (x > 0).mean()), GNG4_detected=("GNG4", lambda x: (x > 0).mean()))
per_donor = per_donor[per_donor.cells >= 20]
per_donor = per_donor.iloc[per_donor.Age.map(age_in_weeks).argsort()]
per_donor.round(2)

,Age,age_class,cells,XCL1_detected,XCL1_mean,XCL2_detected,GNG4_detected
donor,,,,,,,
C41,8w,fetal,49,0.14,0.90,0.06,0.24
F22,9w,fetal,22,0.05,0.28,0.05,0.18
F74,10w,fetal,116,0.25,1.42,0.18,0.31
F23,11w,fetal,44,0.18,0.95,0.11,0.34
F64,11w,fetal,127,0.30,1.71,0.17,0.31
F67,12w,fetal,347,0.29,1.62,0.19,0.37
F45,12w,fetal,105,0.29,1.68,0.19,0.33
F38,13w,fetal,97,0.28,1.48,0.14,0.34
F30,14w,fetal,203,0.31,1.68,0.14,0.42


In [4]:
# Per age class, and the XCL1 / GNG4 overlap named in the claim ("GNG4+ CD8αα+ T(I) cells expressed XCL1")
def summary(d):
    xcl1 = d.XCL1 > 0
    gng4 = d.GNG4 > 0
    return pd.Series({"cells": len(d), "XCL1 detected": xcl1.mean(), "XCL1 mean": d.XCL1.mean(),
                      "XCL2 detected": (d.XCL2 > 0).mean(), "GNG4 detected": gng4.mean(),
                      "XCL1+ among GNG4+": xcl1[gng4].mean(), "XCL1+ among GNG4-": xcl1[~gng4].mean()})
cd8.groupby("age_class").apply(summary).round(2)

,cells,XCL1 detected,XCL1 mean,XCL2 detected,GNG4 detected,XCL1+ among GNG4+,XCL1+ among GNG4-
age_class,,,,,,,
fetal,1469.0,0.29,1.60,0.16,0.35,0.59,0.13
postnatal,2430.0,0.12,0.59,0.05,0.28,0.27,0.06


In [5]:
# Receivers: XCR1 and CCR7 per population and age class
rec = sel[sel.group == "dc1"].join(logexpr)
rec.groupby(["Anno_level_5", "age_class"]).agg(
    cells=("XCR1", "size"), XCR1_detected=("XCR1", lambda x: (x > 0).mean()),
    CCR7_detected=("CCR7", lambda x: (x > 0).mean())).round(2)

cells  XCR1_detected  CCR7_detected
Anno_level_5 age_class                                     
DC1          fetal       1071           0.58           0.04
             postnatal    312           0.49           0.06
aDC1         fetal        100           0.02           0.90
             postnatal     35           0.03           0.89

**Reading.**
- **XCL1 in CD8αα(I) cells drops at birth rather than declining steadily:**
  - fetal donors: 14–43% of cells, apart from F22 at 9 PCW (5%); 29% overall;
  - postnatal donors: 5–21%; 12% overall;
  - the mean level is about three times lower after birth (0.59 against 1.60);
  - after birth there is no further decline: the 13-year-old (18%) and 35-year-old (21%, 24 cells) donors are not
    lower than the 3-month-old (10%).
  - XCL2 follows the same pattern (16% → 5%).
- **Correction to case 2:** the "36% at 7–11 PCW" in case 2 (notebook 05) was measured on the 45 early cells kept by
  the cap. On all cells, the earliest donors (8–11 PCW) are at 5–30%.
- **"GNG4⁺ CD8αα⁺ T(I) cells expressed XCL1" holds, mostly in the fetal thymus:** 59% of GNG4⁺ CD8αα(I) cells
  express XCL1 there, against 13% of GNG4⁻ cells. After birth it is 27% against 6%.
- **Receivers do not change with age:** XCR1 is in about half of DC1 cells and CCR7 in about 90% of aDC1 cells, in
  both age classes.
- The question for the two runs is therefore whether the dependencies of XCL1 (and of the other ligands) inside
  CD8αα(I) cells differ between the fetal and the postnatal thymus.

In [6]:
cols = ["Anno_level_5", "group", "age_class", "donor", "Age", "method", "sort", "Source", "n_counts", "n_genes", "mito"]
sel[cols].rename(columns={"Anno_level_5": "population"}).to_csv(OUT / "selected_cells.csv")
per_donor.round(4).to_csv(OUT / "xcl1_cd8aa_per_donor.csv")
print(len(sel), "cells written to", OUT / "selected_cells.csv")

5417 cells written to ../../results/thymus_case3_xcl1_age/01_cell_populations/selected_cells.csv


## Summary

| | Fetal (7–17 PCW) | Postnatal (3 months – 35 years) |
|---|---|---|
| CD8αα(I) (senders) | 1,469 cells, 14 donors | 2,430 cells, 8 donors |
| DC1 / aDC1 (receivers) | 1,071 / 100 | 312 / 35 |
| XCL1 in CD8αα(I) | 29% of cells (59% of GNG4⁺) | 12% (27% of GNG4⁺) |
| XCR1 in DC1 / CCR7 in aDC1 | 58% / 90% | 49% / 89% |

Next (notebook 02): candidate L–R pairs for both runs.